# 08_02 EES Framework – Fidelity Score

## Purpose

This notebook calculates the **Fidelity Score** for the Explainability-Ethics Scoring (EES) framework.

It is configurable across:

- German Credit
- Pima Diabetes
- COMPAS

Only the `ACTIVE_DATASET` value should be changed.

## Revised Fidelity Definition

Fidelity measures the degree to which explanations accurately and consistently reflect the model's behaviour.

The revised Fidelity pillar uses three equal-weight evidence sources:

```text
Fidelity Score =
1/3 SHAP Reconstruction Fidelity
+ 1/3 LIME Local Surrogate Fidelity
+ 1/3 SHAP-LIME Agreement Score
```

This makes Fidelity stronger because it evaluates:

1. Whether SHAP can reconstruct the model prediction.
2. Whether LIME can locally approximate the model.
3. Whether SHAP and LIME tell a consistent explanation story.

## Component Logic

### 1. SHAP Reconstruction Fidelity

SHAP reconstruction fidelity compares:

```text
model predicted probability
vs
SHAP reconstructed probability
```

It is calculated as:

```text
SHAP Reconstruction Fidelity = 1 - mean reconstruction error
```

Higher is better.

### 2. LIME Local Surrogate Fidelity

LIME fidelity uses the local surrogate model's R² score:

```text
LIME Local Fidelity = mean LIME R² score
```

Higher means the local LIME surrogate approximates the black-box model more closely.

### 3. SHAP-LIME Agreement

SHAP-LIME Agreement uses Kendall's Tau rank agreement converted to a 0–1 scale:

```text
Agreement Score = (Kendall Tau + 1) / 2
```

This measures whether SHAP and LIME rank similar features as important.

### Final Score

All three components are equally weighted to avoid privileging a single explainer.

In [1]:
# =========================================================
# 0. Dataset Configuration
# =========================================================

# Select only one active dataset.
# ACTIVE_DATASET = "german_credit"
# ACTIVE_DATASET = "pima_diabetes"
ACTIVE_DATASET = "compas"

DATASET_CONFIGS = {
    "german_credit": {
        "dataset_name": "german_credit",
        "dataset_label": "German Credit",
        "target_name": "target",
        "positive_class_label": "Default",
        "negative_class_label": "Non-default",
        "results_folder": "german_credit",
        "ees_input_folder": "08_01_ees_framework_prepare_inputs_german_credit",
        "fidelity_folder": "08_02_ees_framework_fidelity_score_german_credit"
    },

    "pima_diabetes": {
        "dataset_name": "pima_diabetes",
        "dataset_label": "Pima Diabetes",
        "target_name": "target",
        "positive_class_label": "Diabetes",
        "negative_class_label": "No diabetes",
        "results_folder": "pima_diabetes",
        "ees_input_folder": "08_01_ees_framework_prepare_inputs_pima_diabetes",
        "fidelity_folder": "08_02_ees_framework_fidelity_score_pima_diabetes"
    },

    "compas": {
        "dataset_name": "compas",
        "dataset_label": "COMPAS",
        "target_name": "target",
        "positive_class_label": "Recidivism",
        "negative_class_label": "No recidivism",
        "results_folder": "compas",
        "ees_input_folder": "08_01_ees_framework_prepare_inputs_compas",
        "fidelity_folder": "08_02_ees_framework_fidelity_score_compas"
    }
}

CONFIG = DATASET_CONFIGS[ACTIVE_DATASET]

print("Active dataset:", CONFIG["dataset_label"])
print("Dataset name:", CONFIG["dataset_name"])

Active dataset: COMPAS
Dataset name: compas


In [2]:
# =========================================================
# 1. Import Required Libraries
# =========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 250)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")
print("Random seed:", RANDOM_STATE)

Libraries imported successfully.
Random seed: 42


In [3]:
# =========================================================
# 2. Mount Google Drive and Configure Paths
# =========================================================

from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount skipped or already available:", e)

ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")

DATASET_NAME = CONFIG["dataset_name"]
NOTEBOOK_NAME = CONFIG["fidelity_folder"]

RESULTS_ROOT = ROOT_PATH / "07_Results" / CONFIG["results_folder"]

EES_INPUT_RESULTS_PATH = RESULTS_ROOT / CONFIG["ees_input_folder"]
EES_INPUT_TABLES_PATH = EES_INPUT_RESULTS_PATH / "tables"
EES_INPUT_DATASETS_PATH = EES_INPUT_RESULTS_PATH / "datasets"

RESULTS_PATH = RESULTS_ROOT / NOTEBOOK_NAME
TABLES_PATH = RESULTS_PATH / "tables"
DATASETS_PATH = RESULTS_PATH / "datasets"
LOGS_PATH = RESULTS_PATH / "logs"

for path in [RESULTS_PATH, TABLES_PATH, DATASETS_PATH, LOGS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Results root:", RESULTS_ROOT)
print("EES input path:", EES_INPUT_RESULTS_PATH)
print("Fidelity output path:", RESULTS_PATH)

Mounted at /content/drive
Results root: /content/drive/MyDrive/Master_Research/07_Results/compas
EES input path: /content/drive/MyDrive/Master_Research/07_Results/compas/08_01_ees_framework_prepare_inputs_compas
Fidelity output path: /content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas


In [4]:
# =========================================================
# 3. Helper Functions
# =========================================================

def read_csv_safely(path, label, required=True):
    path = Path(path)
    if not path.exists():
        if required:
            raise FileNotFoundError(f"{label} not found at: {path}")
        print(f"Optional file not found: {label} at {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def find_first_existing_file(candidate_paths, label, required=True):
    for path in candidate_paths:
        path = Path(path)
        if path.exists():
            print(f"{label} found at: {path}")
            return path

    if required:
        checked = "\n".join(str(path) for path in candidate_paths)
        raise FileNotFoundError(f"{label} not found. Checked:\n{checked}")

    print(f"{label} not found.")
    return None


def save_df(df, filename, folder=None, index=False):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    df.to_csv(output_path, index=index)
    print(f"Saved: {output_path}")
    return output_path


def save_excel(dfs, filename, folder=None):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dfs.items():
            if isinstance(df, pd.DataFrame):
                df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

    print(f"Saved: {output_path}")
    return output_path


def safe_clip_01(series):
    return series.clip(lower=0, upper=1)


def classify_fidelity(score):
    if pd.isna(score):
        return "Not available"
    if score >= 0.80:
        return "High fidelity"
    if score >= 0.60:
        return "Moderate-high fidelity"
    if score >= 0.40:
        return "Moderate fidelity"
    if score >= 0.20:
        return "Low-moderate fidelity"
    return "Low fidelity"

## 4. Load EES Prepared Inputs

This notebook reads the standardised input files created by `08_01`.

In [5]:
# =========================================================
# 4. Load EES Prepared Inputs
# =========================================================

shap_reconstruction_file = find_first_existing_file(
    [
        EES_INPUT_DATASETS_PATH / "ees_shap_reconstruction_input.csv",
        EES_INPUT_TABLES_PATH / "ees_shap_reconstruction_input.csv"
    ],
    "EES SHAP reconstruction input"
)

lime_fidelity_file = find_first_existing_file(
    [
        EES_INPUT_DATASETS_PATH / "ees_lime_fidelity_input.csv",
        EES_INPUT_TABLES_PATH / "ees_lime_fidelity_input.csv"
    ],
    "EES LIME fidelity input"
)

agreement_file = find_first_existing_file(
    [
        EES_INPUT_DATASETS_PATH / "ees_shap_lime_agreement_input.csv",
        EES_INPUT_TABLES_PATH / "ees_shap_lime_agreement_input.csv"
    ],
    "EES SHAP-LIME agreement input"
)

master_ees_input_file = find_first_existing_file(
    [
        EES_INPUT_DATASETS_PATH / "master_ees_input.csv",
        EES_INPUT_TABLES_PATH / "master_ees_input.csv"
    ],
    "Master EES input",
    required=False
)

shap_reconstruction_input_df = read_csv_safely(shap_reconstruction_file, "EES SHAP reconstruction input")
lime_fidelity_input_df = read_csv_safely(lime_fidelity_file, "EES LIME fidelity input")
shap_lime_agreement_input_df = read_csv_safely(agreement_file, "EES SHAP-LIME agreement input")
master_ees_input_df = read_csv_safely(master_ees_input_file, "Master EES input", required=False) if master_ees_input_file else pd.DataFrame()

print("shap_reconstruction_input_df:", shap_reconstruction_input_df.shape)
print("lime_fidelity_input_df:", lime_fidelity_input_df.shape)
print("shap_lime_agreement_input_df:", shap_lime_agreement_input_df.shape)
print("master_ees_input_df:", master_ees_input_df.shape)

display(shap_reconstruction_input_df.head())
display(lime_fidelity_input_df.head())
display(shap_lime_agreement_input_df.head())

EES SHAP reconstruction input found at: /content/drive/MyDrive/Master_Research/07_Results/compas/08_01_ees_framework_prepare_inputs_compas/datasets/ees_shap_reconstruction_input.csv
EES LIME fidelity input found at: /content/drive/MyDrive/Master_Research/07_Results/compas/08_01_ees_framework_prepare_inputs_compas/datasets/ees_lime_fidelity_input.csv
EES SHAP-LIME agreement input found at: /content/drive/MyDrive/Master_Research/07_Results/compas/08_01_ees_framework_prepare_inputs_compas/datasets/ees_shap_lime_agreement_input.csv
Master EES input found at: /content/drive/MyDrive/Master_Research/07_Results/compas/08_01_ees_framework_prepare_inputs_compas/datasets/master_ees_input.csv
shap_reconstruction_input_df: (17500, 14)
lime_fidelity_input_df: (17500, 12)
shap_lime_agreement_input_df: (5, 11)
master_ees_input_df: (5, 25)


,dataset_name,dataset_label,model_id,model_name,test_instance_position,expected_value,shap_sum,reconstructed_raw_output,reconstructed_probability,predicted_probability,reconstruction_error,reconstruction_scale,explainer_type,fallback_used
0,compas,COMPAS,logistic_regression,Logistic Regression,0,-0.102798,-1.024104,-1.126901,0.244733,0.244733,0.000000,log_odds,LinearExplainer,False
1,compas,COMPAS,logistic_regression,Logistic Regression,1,-0.102798,-3.034396,-3.137194,0.041599,0.041599,0.000000,log_odds,LinearExplainer,False
2,compas,COMPAS,logistic_regression,Logistic Regression,2,-0.102798,-0.347893,-0.450690,0.389197,0.389197,0.000000,log_odds,LinearExplainer,False
3,compas,COMPAS,logistic_regression,Logistic Regression,3,-0.102798,0.153230,0.050433,0.512606,0.512606,0.000000,log_odds,LinearExplainer,False
4,compas,COMPAS,logistic_regression,Logistic Regression,4,-0.102798,-1.718524,-1.821322,0.139275,0.139275,0.000000,log_odds,LinearExplainer,False


,dataset_name,dataset_label,model_id,model_name,test_instance_position,predicted_probability,lime_local_prediction,lime_r2_score,lime_local_error,num_features,num_samples,random_state
0,compas,COMPAS,logistic_regression,Logistic Regression,0,0.244733,0.452343,0.227896,0.207610,10,5000,42
1,compas,COMPAS,logistic_regression,Logistic Regression,1,0.041599,0.259893,0.371945,0.218294,10,5000,42
2,compas,COMPAS,logistic_regression,Logistic Regression,2,0.389197,0.210325,0.317731,0.178871,10,5000,42
3,compas,COMPAS,logistic_regression,Logistic Regression,3,0.512606,0.345570,0.302428,0.167036,10,5000,42
4,compas,COMPAS,logistic_regression,Logistic Regression,4,0.139275,0.219065,0.405039,0.079789,10,5000,42


,dataset_name,dataset_label,model_id,model_name,n_instances,mean_kendall_tau,shap_lime_agreement_score,shap_lime_agreement_score_pct,shap_lime_agreement_rank,shap_lime_agreement_band,mean_top_k_overlap_ratio
0,compas,COMPAS,decision_tree,Decision Tree,3500,0.023010,0.511505,51.150489,1,Weak agreement,0.448543
1,compas,COMPAS,random_forest,Random Forest,3500,-0.121791,0.439104,43.910428,2,Weak agreement,0.383429
2,compas,COMPAS,xgboost,XGBoost,3500,-0.129536,0.435232,43.523177,3,Weak agreement,0.428143
3,compas,COMPAS,mlp,MLP,3500,-0.309864,0.345068,34.506810,4,Poor agreement,0.288286
4,compas,COMPAS,logistic_regression,Logistic Regression,3500,-0.326682,0.336659,33.665877,5,Poor agreement,0.341257


## 5. Validate Fidelity Inputs

In [6]:
# =========================================================
# 5. Validate Fidelity Inputs
# =========================================================

if "shap_lime_agreement_score" not in shap_lime_agreement_input_df.columns:
    if "mean_agreement_score" in shap_lime_agreement_input_df.columns:
        shap_lime_agreement_input_df = shap_lime_agreement_input_df.rename(
            columns={"mean_agreement_score": "shap_lime_agreement_score"}
        )

required_shap_cols = ["model_id", "model_name", "reconstruction_error"]
required_lime_cols = ["model_id", "model_name", "lime_r2_score"]
required_agreement_cols = ["model_id", "model_name", "shap_lime_agreement_score"]

validation_rows = []

for label, df, cols in [
    ("SHAP reconstruction", shap_reconstruction_input_df, required_shap_cols),
    ("LIME fidelity", lime_fidelity_input_df, required_lime_cols),
    ("SHAP-LIME agreement", shap_lime_agreement_input_df, required_agreement_cols),
]:
    validation_rows.append({
        "check": f"{label} table not empty",
        "passed": not df.empty
    })
    for col in cols:
        validation_rows.append({
            "check": f"{label} has {col}",
            "passed": col in df.columns
        })

fidelity_input_validation_df = pd.DataFrame(validation_rows)

display(fidelity_input_validation_df)

save_df(fidelity_input_validation_df, "fidelity_input_validation.csv")

if not fidelity_input_validation_df["passed"].all():
    raise ValueError("One or more fidelity input validation checks failed.")

,check,passed
0,SHAP reconstruction table not empty,True
1,SHAP reconstruction has model_id,True
2,SHAP reconstruction has model_name,True
3,SHAP reconstruction has reconstruction_error,True
4,LIME fidelity table not empty,True
5,LIME fidelity has model_id,True
6,LIME fidelity has model_name,True
7,LIME fidelity has lime_r2_score,True
8,SHAP-LIME agreement table not empty,True
9,SHAP-LIME agreement has model_id,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/tables/fidelity_input_validation.csv


## 6. Calculate SHAP Reconstruction Fidelity

The SHAP component is based on reconstruction error.

```text
SHAP Reconstruction Fidelity = 1 - mean reconstruction error
```

A reconstruction error close to zero means SHAP's additive explanation reconstructs the model's predicted probability closely.

In [7]:
# =========================================================
# 6. Calculate SHAP Reconstruction Fidelity
# =========================================================

shap_reconstruction_available_df = shap_reconstruction_input_df.copy()

if "reconstruction_error" not in shap_reconstruction_available_df.columns:
    raise ValueError("reconstruction_error column is required for SHAP reconstruction fidelity.")

shap_reconstruction_summary_df = (
    shap_reconstruction_available_df
    .groupby(["model_id", "model_name"], as_index=False)
    .agg(
        n_shap_rows=("reconstruction_error", "size"),
        n_shap_reconstruction_available=("reconstruction_error", lambda x: int(x.notna().sum())),
        mean_reconstruction_error=("reconstruction_error", "mean"),
        median_reconstruction_error=("reconstruction_error", "median"),
        max_reconstruction_error=("reconstruction_error", "max")
    )
)

shap_reconstruction_summary_df["shap_reconstruction_available_pct"] = (
    shap_reconstruction_summary_df["n_shap_reconstruction_available"] /
    shap_reconstruction_summary_df["n_shap_rows"] * 100
)

shap_reconstruction_summary_df["shap_reconstruction_fidelity_score"] = safe_clip_01(
    1 - shap_reconstruction_summary_df["mean_reconstruction_error"]
)

# If no reconstruction evidence exists for a model, keep score as NaN.
shap_reconstruction_summary_df.loc[
    shap_reconstruction_summary_df["n_shap_reconstruction_available"] == 0,
    "shap_reconstruction_fidelity_score"
] = np.nan

shap_reconstruction_summary_df["shap_reconstruction_fidelity_score_pct"] = (
    shap_reconstruction_summary_df["shap_reconstruction_fidelity_score"] * 100
)

display(shap_reconstruction_summary_df)

save_df(shap_reconstruction_summary_df, "shap_reconstruction_fidelity_summary.csv")

,model_id,model_name,n_shap_rows,n_shap_reconstruction_available,mean_reconstruction_error,median_reconstruction_error,max_reconstruction_error,shap_reconstruction_available_pct,shap_reconstruction_fidelity_score,shap_reconstruction_fidelity_score_pct
0,decision_tree,Decision Tree,3500,3500,0.000000,0.000000,0.000000,100.000000,1.000000,99.999999
1,logistic_regression,Logistic Regression,3500,3500,0.000000,0.000000,0.000000,100.000000,1.000000,100.000000
2,mlp,MLP,3500,3500,0.000000,0.000000,0.000000,100.000000,1.000000,100.000000
3,random_forest,Random Forest,3500,3500,0.000556,0.000067,0.149933,100.000000,0.999444,99.944395
4,xgboost,XGBoost,3500,3500,0.000000,0.000000,0.000000,100.000000,1.000000,99.999996


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/tables/shap_reconstruction_fidelity_summary.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/tables/shap_reconstruction_fidelity_summary.csv')

## 7. Calculate LIME Local Surrogate Fidelity

LIME fidelity uses the average local surrogate R² score.

```text
LIME Local Fidelity = mean LIME R² score
```

Higher values indicate that the LIME local surrogate approximates the black-box model more closely around each instance.

In [8]:
# =========================================================
# 7. Calculate LIME Local Surrogate Fidelity
# =========================================================

lime_fidelity_summary_df = (
    lime_fidelity_input_df
    .groupby(["model_id", "model_name"], as_index=False)
    .agg(
        n_lime_instances=("test_instance_position", "nunique"),
        mean_lime_r2_score=("lime_r2_score", "mean"),
        median_lime_r2_score=("lime_r2_score", "median"),
        min_lime_r2_score=("lime_r2_score", "min"),
        max_lime_r2_score=("lime_r2_score", "max"),
        mean_lime_local_error=("lime_local_error", "mean") if "lime_local_error" in lime_fidelity_input_df.columns else ("lime_r2_score", "mean")
    )
)

lime_fidelity_summary_df["lime_fidelity_score"] = safe_clip_01(
    lime_fidelity_summary_df["mean_lime_r2_score"]
)

lime_fidelity_summary_df["lime_fidelity_score_pct"] = (
    lime_fidelity_summary_df["lime_fidelity_score"] * 100
)

display(lime_fidelity_summary_df)

save_df(lime_fidelity_summary_df, "lime_local_surrogate_fidelity_summary.csv")

,model_id,model_name,n_lime_instances,mean_lime_r2_score,median_lime_r2_score,min_lime_r2_score,max_lime_r2_score,mean_lime_local_error,lime_fidelity_score,lime_fidelity_score_pct
0,decision_tree,Decision Tree,3500,0.263802,0.276838,0.062579,0.466154,0.103848,0.263802,26.380195
1,logistic_regression,Logistic Regression,3500,0.366766,0.339854,0.027434,0.736540,0.080377,0.366766,36.676593
2,mlp,MLP,3500,0.456811,0.454438,0.065053,0.671418,0.138946,0.456811,45.681116
3,random_forest,Random Forest,3500,0.404200,0.442965,0.122090,0.554774,0.291376,0.404200,40.419980
4,xgboost,XGBoost,3500,0.518743,0.560029,0.163269,0.705494,0.125687,0.518743,51.874272


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/tables/lime_local_surrogate_fidelity_summary.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/tables/lime_local_surrogate_fidelity_summary.csv')

## 8. Prepare SHAP-LIME Agreement Fidelity Component

This component comes from Notebook `06`.

It captures whether SHAP and LIME rank similar features as important.

In [9]:
# =========================================================
# 8. Prepare SHAP-LIME Agreement Fidelity Component
# =========================================================

agreement_component_df = shap_lime_agreement_input_df.copy()

if "shap_lime_agreement_score" not in agreement_component_df.columns:
    if "mean_agreement_score" in agreement_component_df.columns:
        agreement_component_df = agreement_component_df.rename(
            columns={"mean_agreement_score": "shap_lime_agreement_score"}
        )
    else:
        raise ValueError("shap_lime_agreement_score is required for the agreement component.")

agreement_component_cols = [
    col for col in [
        "model_id",
        "model_name",
        "n_instances",
        "mean_kendall_tau",
        "shap_lime_agreement_score",
        "shap_lime_agreement_score_pct",
        "mean_top_k_overlap_ratio"
    ]
    if col in agreement_component_df.columns
]

agreement_component_df = agreement_component_df[agreement_component_cols].copy()

agreement_component_df["shap_lime_agreement_score"] = safe_clip_01(
    agreement_component_df["shap_lime_agreement_score"]
)

if "shap_lime_agreement_score_pct" not in agreement_component_df.columns:
    agreement_component_df["shap_lime_agreement_score_pct"] = (
        agreement_component_df["shap_lime_agreement_score"] * 100
    )

display(agreement_component_df)

save_df(agreement_component_df, "shap_lime_agreement_fidelity_component.csv")

,model_id,model_name,n_instances,mean_kendall_tau,shap_lime_agreement_score,shap_lime_agreement_score_pct,mean_top_k_overlap_ratio
0,decision_tree,Decision Tree,3500,0.023010,0.511505,51.150489,0.448543
1,random_forest,Random Forest,3500,-0.121791,0.439104,43.910428,0.383429
2,xgboost,XGBoost,3500,-0.129536,0.435232,43.523177,0.428143
3,mlp,MLP,3500,-0.309864,0.345068,34.506810,0.288286
4,logistic_regression,Logistic Regression,3500,-0.326682,0.336659,33.665877,0.341257


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/tables/shap_lime_agreement_fidelity_component.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/tables/shap_lime_agreement_fidelity_component.csv')

## 9. Calculate Final Equal-Weight Fidelity Score

The final Fidelity score is calculated as:

```text
Fidelity =
1/3 SHAP Reconstruction Fidelity
+ 1/3 LIME Local Surrogate Fidelity
+ 1/3 SHAP-LIME Agreement Score
```

A score close to 1 indicates that the model's explanations are faithful, locally approximable and consistent across explainers.

In [10]:
# =========================================================
# 9. Calculate Final Equal-Weight Fidelity Score
# =========================================================

fidelity_score_df = shap_reconstruction_summary_df[
    [
        "model_id",
        "model_name",
        "shap_reconstruction_fidelity_score",
        "mean_reconstruction_error",
        "shap_reconstruction_available_pct"
    ]
].copy()

fidelity_score_df = fidelity_score_df.merge(
    lime_fidelity_summary_df[
        [
            "model_id",
            "lime_fidelity_score",
            "mean_lime_r2_score",
            "mean_lime_local_error"
        ]
    ],
    on="model_id",
    how="outer"
)

fidelity_score_df = fidelity_score_df.merge(
    agreement_component_df[
        [
            "model_id",
            "shap_lime_agreement_score"
        ] + (["mean_kendall_tau"] if "mean_kendall_tau" in agreement_component_df.columns else [])
    ],
    on="model_id",
    how="outer"
)

# Restore model_name where outer joins created gaps.
if "model_name" not in fidelity_score_df.columns or fidelity_score_df["model_name"].isna().any():
    model_name_lookup = pd.concat(
        [
            shap_reconstruction_summary_df[["model_id", "model_name"]],
            lime_fidelity_summary_df[["model_id", "model_name"]],
            agreement_component_df[["model_id", "model_name"]] if "model_name" in agreement_component_df.columns else pd.DataFrame(columns=["model_id", "model_name"])
        ],
        ignore_index=True
    ).drop_duplicates("model_id")

    fidelity_score_df = fidelity_score_df.drop(columns=["model_name"], errors="ignore").merge(
        model_name_lookup,
        on="model_id",
        how="left"
    )

component_cols = [
    "shap_reconstruction_fidelity_score",
    "lime_fidelity_score",
    "shap_lime_agreement_score"
]

fidelity_score_df["n_fidelity_components_available"] = fidelity_score_df[component_cols].notna().sum(axis=1)

fidelity_score_df["fidelity_score"] = fidelity_score_df[component_cols].mean(axis=1)

fidelity_score_df["fidelity_score_pct"] = fidelity_score_df["fidelity_score"] * 100

fidelity_score_df["fidelity_rank"] = (
    fidelity_score_df["fidelity_score"]
    .rank(ascending=False, method="dense")
    .astype(int)
)

fidelity_score_df["fidelity_band"] = fidelity_score_df["fidelity_score"].apply(classify_fidelity)

fidelity_score_df["fidelity_evidence_note"] = fidelity_score_df.apply(
    lambda row: (
        "All three fidelity components available."
        if row["n_fidelity_components_available"] == 3
        else f"Only {int(row['n_fidelity_components_available'])} of 3 fidelity components available."
    ),
    axis=1
)

fidelity_score_df = (
    fidelity_score_df
    .sort_values(["fidelity_rank", "model_id"])
    .reset_index(drop=True)
)

# Add dataset metadata.
fidelity_score_df.insert(0, "dataset_name", DATASET_NAME)
fidelity_score_df.insert(1, "dataset_label", CONFIG["dataset_label"])

display(fidelity_score_df)

save_df(fidelity_score_df, "ees_fidelity_model_scores.csv")
save_df(fidelity_score_df, "ees_fidelity_model_scores.csv", folder=DATASETS_PATH)

,dataset_name,dataset_label,model_id,model_name,shap_reconstruction_fidelity_score,mean_reconstruction_error,shap_reconstruction_available_pct,lime_fidelity_score,mean_lime_r2_score,mean_lime_local_error,shap_lime_agreement_score,mean_kendall_tau,n_fidelity_components_available,fidelity_score,fidelity_score_pct,fidelity_rank,fidelity_band,fidelity_evidence_note
0,compas,COMPAS,xgboost,XGBoost,1.000000,0.000000,100.000000,0.518743,0.518743,0.125687,0.435232,-0.129536,3,0.651325,65.132482,1,Moderate-high fidelity,All three fidelity components available.
1,compas,COMPAS,random_forest,Random Forest,0.999444,0.000556,100.000000,0.404200,0.404200,0.291376,0.439104,-0.121791,3,0.614249,61.424934,2,Moderate-high fidelity,All three fidelity components available.
2,compas,COMPAS,mlp,MLP,1.000000,0.000000,100.000000,0.456811,0.456811,0.138946,0.345068,-0.309864,3,0.600626,60.062642,3,Moderate-high fidelity,All three fidelity components available.
3,compas,COMPAS,decision_tree,Decision Tree,1.000000,0.000000,100.000000,0.263802,0.263802,0.103848,0.511505,0.023010,3,0.591769,59.176894,4,Moderate fidelity,All three fidelity components available.
4,compas,COMPAS,logistic_regression,Logistic Regression,1.000000,0.000000,100.000000,0.366766,0.366766,0.080377,0.336659,-0.326682,3,0.567808,56.780823,5,Moderate fidelity,All three fidelity components available.


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/tables/ees_fidelity_model_scores.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/datasets/ees_fidelity_model_scores.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/datasets/ees_fidelity_model_scores.csv')

## 10. Fidelity Component Weighting Table

In [11]:
# =========================================================
# 10. Fidelity Component Weighting Table
# =========================================================

fidelity_component_weighting_df = pd.DataFrame([
    {
        "component": "SHAP Reconstruction Fidelity",
        "score_column": "shap_reconstruction_fidelity_score",
        "weight": 1/3,
        "purpose": "Measures whether SHAP reconstructs the model predicted probability."
    },
    {
        "component": "LIME Local Surrogate Fidelity",
        "score_column": "lime_fidelity_score",
        "weight": 1/3,
        "purpose": "Measures whether LIME locally approximates the black-box model."
    },
    {
        "component": "SHAP-LIME Agreement",
        "score_column": "shap_lime_agreement_score",
        "weight": 1/3,
        "purpose": "Measures whether independent explainers provide consistent feature rankings."
    }
])

fidelity_component_weighting_df["weight_pct"] = fidelity_component_weighting_df["weight"] * 100

display(fidelity_component_weighting_df)

save_df(fidelity_component_weighting_df, "fidelity_component_weighting.csv")

,component,score_column,weight,purpose,weight_pct
0,SHAP Reconstruction Fidelity,shap_reconstruction_fidelity_score,0.333333,Measures whether SHAP reconstructs the model p...,33.333333
1,LIME Local Surrogate Fidelity,lime_fidelity_score,0.333333,Measures whether LIME locally approximates the...,33.333333
2,SHAP-LIME Agreement,shap_lime_agreement_score,0.333333,Measures whether independent explainers provid...,33.333333


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/tables/fidelity_component_weighting.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/tables/fidelity_component_weighting.csv')

## 11. Fidelity Interpretation Table

In [12]:
# =========================================================
# 11. Fidelity Interpretation Table
# =========================================================

fidelity_interpretation_table_df = pd.DataFrame([
    {
        "score_range": "0.80 to 1.00",
        "band": "High fidelity",
        "interpretation": "Explanations strongly reconstruct, approximate and agree across methods."
    },
    {
        "score_range": "0.60 to <0.80",
        "band": "Moderate-high fidelity",
        "interpretation": "Explanations are generally faithful, with some limitations."
    },
    {
        "score_range": "0.40 to <0.60",
        "band": "Moderate fidelity",
        "interpretation": "Explanation faithfulness is mixed and should be interpreted cautiously."
    },
    {
        "score_range": "0.20 to <0.40",
        "band": "Low-moderate fidelity",
        "interpretation": "Explanations show weak faithfulness and require caution."
    },
    {
        "score_range": "0.00 to <0.20",
        "band": "Low fidelity",
        "interpretation": "Explanations poorly reflect model behaviour."
    }
])

display(fidelity_interpretation_table_df)

save_df(fidelity_interpretation_table_df, "fidelity_interpretation_table.csv")

,score_range,band,interpretation
0,0.80 to 1.00,High fidelity,"Explanations strongly reconstruct, approximate..."
1,0.60 to <0.80,Moderate-high fidelity,"Explanations are generally faithful, with some..."
2,0.40 to <0.60,Moderate fidelity,Explanation faithfulness is mixed and should b...
3,0.20 to <0.40,Low-moderate fidelity,Explanations show weak faithfulness and requir...
4,0.00 to <0.20,Low fidelity,Explanations poorly reflect model behaviour.


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/tables/fidelity_interpretation_table.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/tables/fidelity_interpretation_table.csv')

## 12. Save Workbook and Manifest

In [13]:
# =========================================================
# 12. Save Workbook and Manifest
# =========================================================

save_excel(
    {
        "fidelity_scores": fidelity_score_df,
        "shap_reconstruction": shap_reconstruction_summary_df,
        "lime_fidelity": lime_fidelity_summary_df,
        "agreement_component": agreement_component_df,
        "component_weighting": fidelity_component_weighting_df,
        "interpretation_table": fidelity_interpretation_table_df,
        "input_validation": fidelity_input_validation_df
    },
    f"ees_fidelity_score_{DATASET_NAME}_outputs.xlsx"
)

manifest = {
    "dataset_name": DATASET_NAME,
    "dataset_label": CONFIG["dataset_label"],
    "notebook_name": NOTEBOOK_NAME,
    "run_timestamp": datetime.now().isoformat(),
    "random_state": RANDOM_STATE,
    "fidelity_formula": "mean of SHAP reconstruction fidelity, LIME local surrogate fidelity and SHAP-LIME agreement score",
    "component_weights": {
        "shap_reconstruction_fidelity": 1/3,
        "lime_local_surrogate_fidelity": 1/3,
        "shap_lime_agreement": 1/3
    },
    "standardisation_note": (
        "This Fidelity notebook uses the same configurable pipeline and equal-weight three-component "
        "Fidelity design across German Credit, Pima Diabetes and COMPAS."
    )
}

with open(LOGS_PATH / "ees_fidelity_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

with open(TABLES_PATH / "ees_fidelity_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print("Workbook and manifest saved.")

Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/tables/ees_fidelity_score_compas_outputs.xlsx
Workbook and manifest saved.


## 13. Interpretation Note

This notebook operationalises Fidelity as a composite measure of explanation faithfulness.

The key contribution is that Fidelity is no longer dependent on a single explainer. Instead, it combines:

- SHAP reconstruction evidence,
- LIME local approximation evidence,
- SHAP-LIME cross-method consistency evidence.

This makes the Fidelity pillar more robust and more defensible for the EES framework.

In [14]:
# =========================================================
# 13. Final Completion Message
# =========================================================

print("=" * 100)
print(f"08_02 EES Fidelity Score completed successfully for: {CONFIG['dataset_label']}")
print("=" * 100)

print(f"Tables saved to:   {TABLES_PATH}")
print(f"Datasets saved to: {DATASETS_PATH}")
print(f"Logs saved to:     {LOGS_PATH}")

print("Key outputs:")
print("- ees_fidelity_model_scores.csv")
print("- shap_reconstruction_fidelity_summary.csv")
print("- lime_local_surrogate_fidelity_summary.csv")
print("- shap_lime_agreement_fidelity_component.csv")
print("- fidelity_component_weighting.csv")
print(f"- ees_fidelity_score_{DATASET_NAME}_outputs.xlsx")

08_02 EES Fidelity Score completed successfully for: COMPAS
Tables saved to:   /content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/tables
Datasets saved to: /content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/datasets
Logs saved to:     /content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/logs
Key outputs:
- ees_fidelity_model_scores.csv
- shap_reconstruction_fidelity_summary.csv
- lime_local_surrogate_fidelity_summary.csv
- shap_lime_agreement_fidelity_component.csv
- fidelity_component_weighting.csv
- ees_fidelity_score_compas_outputs.xlsx
